In [1]:
import pickle
import os
import ao_core as ao
import ao_arch as ar
import numpy as np

In [2]:
description = "Basic MNIST"
arch_i = [28 * 28]
arch_z = [4]
arch_c = []
connector_function = "rand_conn"

connector_parameters = [392, 261, 784, 4]
arch = ar.Arch( arch_i, arch_z, arch_c, connector_function, connector_parameters, description)\

# connector_function = "full_conn"  
# arch = ar.Arch(arch_i, arch_z, arch_c, connector_function, description)

agent = ao.Agent(arch, notes="B&W MNIST Agent", save_meta=False)



In [3]:
data_dir = 'D:/AO/MNIST/loss-of-plasticity/lop/permuted_mnist/data/permutated_data/'
task_files = sorted([os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith('.pkl')])

In [4]:
def down_sample(image, down=200):
    down_image = np.zeros(image.shape)
    down_image[image < down] = 0
    down_image[image >= down] = 1
    return down_image

def process_labels(labels):
    labels = np.array(labels) 
    label_to_binary = np.zeros([10, 4], dtype="int8")
    for i in np.arange(10):
        label_to_binary[i] = np.array(list(np.binary_repr(i, 4)), dtype=int)

    # Changing the labels from 0-9 int to binary for our weightless neural state machine
    labels_z = np.zeros([labels.size, 4])
    for i in np.arange(labels.size):
        labels_z[i] = label_to_binary[labels[i]]
    return labels_z

for task_idx, task_file in enumerate(task_files[0:1]):
    
    with open(task_file, 'rb') as f:
        permuted_x, permuted_y = pickle.load(f)
    permuted_x = np.array(permuted_x)
    permuted_x = permuted_x * (255.0)
    permuted_x = permuted_x.astype(int).reshape(len(permuted_x), 28, 28)
    images = down_sample(permuted_x).reshape(len(permuted_x),784)
    

    permuted_y_binary = process_labels(permuted_y)
    permuted_y_binary = np.array(permuted_y_binary)
    permuted_y_binary = permuted_y_binary.astype(float)

    
    #training
    for index, image in enumerate(images[0:10]):
        INPUT = image
        LABEL = permuted_y_binary[index]
        print(LABEL)
        agent.reset_state()
        agent.next_state(INPUT, LABEL, DD=False, unsequenced=True)
    

    print('Training done')
    #testing
    for index, image in enumerate(images[0:10]):
        INPUT = image
        LABEL = permuted_y_binary[index]
    
        for i in range(10): 
            result = agent.next_state(INPUT, DD=False, unsequenced=True)
            # print(result)
        
        print(LABEL, result)
        if np.array_equal(LABEL, result):
            print('Predicted correctly')



C:\Users\kusha\AppData\Local\Temp\ipykernel_23868\2805721115.py:23: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  permuted_x = np.array(permuted_x)
C:\Users\kusha\AppData\Local\Temp\ipykernel_23868\2805721115.py:8: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  labels = np.array(labels)


[0. 1. 1. 0.]
[0. 1. 1. 1.]
[0. 1. 1. 0.]
[0. 1. 1. 1.]
[0. 0. 0. 1.]
[0. 1. 1. 1.]
[0. 0. 0. 0.]
[0. 0. 0. 0.]
[0. 1. 1. 0.]
[0. 1. 1. 1.]
Training done
[0. 1. 1. 0.] [0 0 0 1]
[0. 1. 1. 1.] [0 0 1 0]
[0. 1. 1. 0.] [1 1 1 0]
[0. 1. 1. 1.] [0 0 1 0]
[0. 0. 0. 1.] [0 0 0 0]
[0. 1. 1. 1.] [0 1 0 1]
[0. 0. 0. 0.] [1 0 1 0]
[0. 0. 0. 0.] [0 0 0 1]
[0. 1. 1. 0.] [1 0 0 1]
[0. 1. 1. 1.] [0 1 1 0]
